<a href="https://colab.research.google.com/github/prabhleenkaursaini/UCS420/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:


• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if d%3 gives "account", write a question like “how do I update my registered mobile number”).


• # Example roll number ...23 -> digits 2, 3


• # digit 2 -> category[2 % 3] = general


• # digit 3 -> category[3 % 3] = billing


Output: Print your final 6-row DataFrame.


In [ ]:
import pandas as pd

roll = "1024170047"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"}
]

last_two = roll[-2:]
categories = ["billing", "account", "general"]

entry1 = {
    "question": "how do i update my registered mobile number",
    "answer": "Go to Settings and update your registered mobile number.",
    "keywords": "mobile number update phone",
    "category": categories[int(last_two[0]) % 3]
}

entry2 = {
    "question": "how do i change my account email",
    "answer": "Go to Account Settings and change your registered email.",
    "keywords": "email account change update",
    "category": categories[int(last_two[1]) % 3]
}

df = pd.DataFrame(fixed_entries + [entry1, entry2])

print(df)


                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5             how do i change my account email   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Settings and update your registered mobi...   
5  Go to Account Settings and change your registe...   

                      keywords category  
0        fee cost price charge  billing  
1         password reset login  account  
2       hours timing open time  general  
3          pay payment upi fee  billing  
4   mobile number update phone  account  
5  e

Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence


In [ ]:
def score_query(query, df):
    query_words = query.lower().split()
    results = []

    for i in range(len(df)):
        keywords = df.loc[i, "keywords"].lower().split()
        score = 0

        for word in query_words:
            if word in keywords:
                score = score + 1

        if score > 0:
            results.append((i, score))

    results.sort(key=lambda x: x[1], reverse=True)

    for i, score in results:
        print("Confidence:", score)
        print(df.loc[i])
        print()

query = input("Enter your question: ")
score_query(query, df)


Enter your question: fee
Confidence: 1
question       what is the annual fee
answer      The annual fee is Rs 500.
keywords        fee cost price charge
category                      billing
Name: 0, dtype: object

Confidence: 1
question                         how can i pay the fee
answer      You can pay via UPI, card, or net banking.
keywords                           pay payment upi fee
category                                       billing
Name: 3, dtype: object



Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.


In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

category_name = entry1["category"]

print("Category:", category_name)
print(same_category(category_name, df))


Category: account
1                          how to reset password
4    how do i update my registered mobile number
5               how do i change my account email
Name: question, dtype: object


Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.


In [ ]:
new_keyword = input("Enter a new keyword: ")

df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_keyword

file_name = roll + "_faq_data.csv"
df.to_csv(file_name, index=False)

print(df)
print("Saved as:", file_name)


Enter a new keyword: annual
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5             how do i change my account email   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Settings and update your registered mobi...   
5  Go to Account Settings and change your registe...   

                       keywords category  
0  fee cost price charge annual  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4    mobile n

Q5: Using groupby, print how many FAQ entries you have per category.


In [ ]:
category_count = df.groupby("category")["question"].count()

print(category_count)


category
account    3
billing    2
general    1
Name: question, dtype: int64


Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.


In [ ]:
def score_query_tie(query, df):
    query_words = query.lower().split()
    results = []

    for i in range(len(df)):
        keywords = df.loc[i, "keywords"].lower().split()
        score = 0

        for word in query_words:
            if word in keywords:
                score = score + 1

        if score > 0:
            results.append((i, score))

    results.sort(key=lambda x: x[1], reverse=True)

    if len(results) == 0:
        print("No matching entry found")
        return

    highest_score = results[0][1]

    for i, score in results:
        if score == highest_score:
            print("Confidence:", score)
            print(df.loc[i])
            print()

print("Query with tie:")
score_query_tie("fee", df)

print("Query without tie:")
score_query_tie("password reset", df)


Query with tie:
Confidence: 1
question          what is the annual fee
answer         The annual fee is Rs 500.
keywords    fee cost price charge annual
category                         billing
Name: 0, dtype: object

Confidence: 1
question                         how can i pay the fee
answer      You can pay via UPI, card, or net banking.
keywords                           pay payment upi fee
category                                       billing
Name: 3, dtype: object

Query without tie:
Confidence: 2
question               how to reset password
answer      Go to Settings > Reset Password.
keywords                password reset login
category                             account
Name: 1, dtype: object

